## Google Colab and data paths

Set `KDD_PROJECT_ROOT` if your project is not stored at `/content/drive/MyDrive/kdd_dropout_project`. The configured folder must contain `dataset/`, `shared/metadata/`, and `artifacts/`.


In [ ]:
from pathlib import Path
import os

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    pass

def discover_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset").exists() and (candidate / "shared" / "metadata").exists():
            return candidate
    return start

default_root = (
    Path("/content/drive/MyDrive/kdd_dropout_project")
    if IN_COLAB else discover_project_root(Path.cwd())
)
PROJECT_ROOT = Path(os.environ.get("KDD_PROJECT_ROOT", default_root))
DATA_ROOT = PROJECT_ROOT / "dataset"
METADATA_ROOT = PROJECT_ROOT / "shared" / "metadata"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODEL_ARTIFACT_ROOT = ARTIFACT_ROOT / "fslsm_bilstm"
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Model artifacts: {MODEL_ARTIFACT_ROOT}")


# Data Preprocessing + Window-Level FSLSM Scoring

Extends the original `02_Data_Preprocessing.ipynb` pipeline. Everything through
sliding-window construction is unchanged; the addition is a per-window rule-based
FSLSM score (Input, Processing, Perception, Understanding), computed from the same
event/category/time-gap slice used to build each training window, saved alongside
the sequence tensors.

In [ ]:
import pandas as pd
import numpy as np
import torch
import os

print("Loading raw data...")
log_train = pd.read_csv(DATA_ROOT / "train/train/log_train.csv")
log_test = pd.read_csv(DATA_ROOT / "test/test/log_test.csv")
object_df = pd.read_csv(METADATA_ROOT / "object.csv")
truth_train = pd.read_csv(DATA_ROOT / "train/train/truth_train.csv", header=None, names=["student_id", "dropout"])
truth_test = pd.read_csv(DATA_ROOT / "test/test/truth_test.csv", header=None, names=["student_id", "dropout"])

log_train['time'] = pd.to_datetime(log_train['time'])
log_test['time'] = pd.to_datetime(log_test['time'])

# ==========================================
# 1. FILTER FIRST 2 WEEKS
# ==========================================
def filter_first_2_weeks(df):
    first_time = df.groupby("enrollment_id")["time"].transform("min")
    return df[(df["time"] - first_time).dt.total_seconds() <= 14 * 24 * 3600].copy()

log_train = filter_first_2_weeks(log_train)
log_test = filter_first_2_weeks(log_test)

# ==========================================
# 2. MERGE OBJECT CATEGORY
# ==========================================
log_train = log_train.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')
log_test = log_test.merge(object_df[['module_id', 'category']], how='left', left_on='object', right_on='module_id')

log_train['category'] = log_train['category'].fillna("unknown")
log_test['category'] = log_test['category'].fillna("unknown")


In [ ]:
# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
def process_features(df):
    df = df.sort_values(['enrollment_id', 'time'])
    df['time_gap'] = np.log1p(df.groupby('enrollment_id')['time'].diff().dt.total_seconds().fillna(0))
    df['token'] = df['source'] + "_" + df['event'] + "_" + df['category']
    return df

log_train = process_features(log_train)
log_test = process_features(log_test)

def build_sequences(df, truth_df):
    tokens = df.groupby("enrollment_id")["token"].apply(list)
    gaps = df.groupby("enrollment_id")["time_gap"].apply(list)
    seq_df = pd.DataFrame({"student_id": tokens.index, "tokens": tokens.values, "gaps": gaps.values})
    return seq_df.merge(truth_df, how="inner", on="student_id")

train_df = build_sequences(log_train, truth_train)
test_df = build_sequences(log_test, truth_test)


In [ ]:
# ==========================================
# 4. ENCODING
# ==========================================
def split_token(token):
    parts = token.split("_")
    return parts if len(parts) == 3 else ["unknown", "unknown", "unknown"]

source_set, event_set, category_set = set(), set(), set()
for seq in train_df["tokens"]:
    for token in seq:
        s, e, c = split_token(token)
        source_set.add(s); event_set.add(e); category_set.add(c)

source_vocab = {v: i+1 for i, v in enumerate(source_set)}
event_vocab = {v: i+1 for i, v in enumerate(event_set)}
category_vocab = {v: i+1 for i, v in enumerate(category_set)}

# Inverse vocabs -- needed to decode encoded ids back to strings inside each
# sliding window so FSLSM group membership (event/category name) can be checked.
inv_event_vocab = {i: v for v, i in event_vocab.items()}
inv_category_vocab = {i: v for v, i in category_vocab.items()}

def apply_encoding(df):
    def encode_multi(seq):
        src, evt, cat = [], [], []
        for token in seq:
            s, e, c = split_token(token)
            src.append(source_vocab.get(s, 0)); evt.append(event_vocab.get(e, 0)); cat.append(category_vocab.get(c, 0))
        return src, evt, cat

    encoded = df["tokens"].apply(encode_multi)
    df["src"], df["evt"], df["cat"] = encoded.str[0], encoded.str[1], encoded.str[2]
    return df

train_df, test_df = apply_encoding(train_df), apply_encoding(test_df)


## FSLSM Window Scoring

Rule-based scores, one `[-1, 1]` value per FSLSM dimension per *window* (not per
learner), following the pattern from `Binary_Visual_Verbal_Scoring.pdf`:

`score = (countA - countB) / (countA + countB)`, extended to all four dimensions
using the event/category groupings from the FSLSM feature-mapping table.

**Approximations used here** (documented so they can be refined later):
- **Input** (Visual vs Verbal): direct event/category match, as in the reference doc.
- **Processing** (Active vs Reflective): Active = `problem`/`discussion` counts;
  Reflective = the summed time-gap (dwell proxy) preceding `video`/`access-html`
  events, using the `time_gap` feature already computed per event.
- **Perception** (Sensing vs Intuitive): Sensing = `problem` event count (repeat-attempt
  proxy); Intuitive = `navigate`→`html` count (broad non-linear browsing proxy).
- **Understanding** (Sequential vs Global): Sequential = `navigate` into
  `chapter`/`sequential`/`vertical` categories; Global = `page_close` events plus
  any other `navigate` events (jump proxy).

These proxies avoid needing extra passes over the raw logs beyond what
`build_windows` already slices, so scoring stays free at window-construction time.

In [ ]:
def compute_window_fslsm(evt_ids, cat_ids, gaps, inv_event_vocab, inv_category_vocab):
    """Compute the 4 FSLSM ratio scores for one sliding window.
    evt_ids, cat_ids, gaps are equal-length lists/arrays for the window (padding = evt_id 0).
    Returns [input_score, processing_score, perception_score, understanding_score].
    """
    visual = verbal = 0.0
    active = reflective = 0.0
    sensing = intuitive = 0.0
    sequential = global_ = 0.0

    for e_id, c_id, gap in zip(evt_ids, cat_ids, gaps):
        if e_id == 0:
            continue  # padding position, skip
        e = inv_event_vocab.get(e_id, "unknown")
        c = inv_category_vocab.get(c_id, "unknown")

        # Input: Visual vs Verbal
        if e == "video" or (e == "access" and c == "video"):
            visual += 1
        if e == "wiki" or e == "discussion" or (e == "access" and c == "html"):
            verbal += 1

        # Processing: Active vs Reflective
        if e == "problem" or e == "discussion":
            active += 1
        if e == "video" or (e == "access" and c == "html"):
            reflective += max(gap, 0.0)  # dwell-time proxy, already log1p-scaled

        # Perception: Sensing vs Intuitive
        if e == "problem":
            sensing += 1
        if e == "navigate" and c == "html":
            intuitive += 1

        # Understanding: Sequential vs Global
        if e == "navigate" and c in ("chapter", "sequential", "vertical"):
            sequential += 1
        if e == "page_close" or (e == "navigate" and c not in ("chapter", "sequential", "vertical")):
            global_ += 1

    def ratio(a, b):
        return (a - b) / (a + b) if (a + b) > 0 else 0.0

    return [
        ratio(visual, verbal),
        ratio(active, reflective),
        ratio(sensing, intuitive),
        ratio(sequential, global_),
    ]


In [ ]:
# ==========================================
# 5. SLIDING WINDOWS (+ FSLSM SCORES)
# ==========================================
MAX_LEN, STRIDE = 150, 75

def build_windows(df, inv_event_vocab, inv_category_vocab):
    src_all, evt_all, cat_all, gap_all, y_all, ids_all, fslsm_all = [], [], [], [], [], [], []
    for _, row in df.iterrows():
        src, evt, cat, gaps = row["src"], row["evt"], row["cat"], row["gaps"]
        for i in range(0, len(src), STRIDE):
            s, e, c, g = src[i:i+MAX_LEN], evt[i:i+MAX_LEN], cat[i:i+MAX_LEN], gaps[i:i+MAX_LEN]
            if not s:
                continue

            # FSLSM scores computed BEFORE padding, from the real (unpadded) window content
            fslsm_scores = compute_window_fslsm(e, c, g, inv_event_vocab, inv_category_vocab)

            pad_len = MAX_LEN - len(s)
            s = s + [0]*pad_len; e = e + [0]*pad_len; c = c + [0]*pad_len; g = g + [0]*pad_len

            src_all.append(s); evt_all.append(e); cat_all.append(c); gap_all.append(g)
            fslsm_all.append(fslsm_scores)
            ids_all.append(row["student_id"])
            y_all.append(row["dropout"])

    return (np.array(src_all), np.array(evt_all), np.array(cat_all), np.array(gap_all),
            np.array(y_all), np.array(ids_all), np.array(fslsm_all, dtype=np.float32))

print("Building Windows (with FSLSM scores)...")
src_train, evt_train, cat_train, gap_train, y_train, _, fslsm_train = build_windows(
    train_df, inv_event_vocab, inv_category_vocab)
src_test, evt_test, cat_test, gap_test, y_test, test_ids, fslsm_test = build_windows(
    test_df, inv_event_vocab, inv_category_vocab)

print(f"Train windows: {src_train.shape[0]}, FSLSM score shape: {fslsm_train.shape}")
print("Mean FSLSM scores (train) [Input, Processing, Perception, Understanding]:")
print(fslsm_train.mean(axis=0))


In [ ]:
# ==========================================
# 6. SAVE PROCESSED DATA
# ==========================================
MODEL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
torch.save({
    "train": (src_train, evt_train, cat_train, gap_train, y_train, fslsm_train),
    "test": (src_test, evt_test, cat_test, gap_test, y_test, test_ids, fslsm_test),
    "vocabs": (source_vocab, event_vocab, category_vocab),
    "truth_test": truth_test
}, MODEL_ARTIFACT_ROOT / "processed_data_fslsm.pt")

print(f"Success! Saved to {MODEL_ARTIFACT_ROOT / 'processed_data_fslsm.pt'}")
print("Note: this file has the SAME src/evt/cat/gap/y/ids as the original processed_data.pt,")
print("plus one new 'fslsm' array per split -- so row 1 (baseline) and row 8 (with-FSLSM)")
print("in the comparison table are trained on identical windows/splits, differing only")
print("in whether the model consumes the extra 4 columns.")
